# 🏙️ Multimodal Urban Event Prediction
### Setup & Environment Verification Notebook

**Project:** Predicting urban demand/events using NYC Taxi data, social media sentiment, weather, and city events  
**Author:** [Your Name]  
**Date:** April 2026

---
This notebook verifies the environment, loads all datasets, and performs initial exploratory data analysis (EDA).

## 1. 环境验证 (Environment Check)

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import geopandas as gpd
import pyarrow
import requests
import holidays
from pathlib import Path
from datetime import datetime

# ── ML / NLP libraries ───────────────────────────────────────────────
import sklearn
from sklearn.preprocessing import StandardScaler

# Optional: if you have transformers installed
try:
    import transformers
    print(f"✅ transformers: {transformers.__version__}")
except ImportError:
    print("⚠️  transformers not installed (install later for sentiment model)")

# ── Version printout ─────────────────────────────────────────────────
print("="*50)
print("✅ Environment Check Passed")
print("="*50)
print(f"  pandas      : {pd.__version__}")
print(f"  numpy       : {np.__version__}")
print(f"  geopandas   : {gpd.__version__}")
print(f"  pyarrow     : {pyarrow.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  holidays    : {holidays.__version__}")
print("="*50)

## 2. 路径配置 (Path Configuration)
> 修改下方路径为你本地的实际路径

In [ ]:
# ── 修改这里的路径 ──────────────────────────────────────────────────
DATA_DIR = Path("../data")   # 相对路径（推荐，把数据复制到 data/ 文件夹）

TAXI_PATH      = DATA_DIR / "yellow_tripdata_2026-01.parquet"
TAXI_ZONES_DIR = DATA_DIR / "taxi_zones"
EVENTS_PATH    = DATA_DIR / "NYCHA_Citywide_Special_Events_20260413.csv"
REDDIT_PATH    = DATA_DIR / "Reddit_Data.csv"
TWITTER_PATH   = DATA_DIR / "Twitter_Data.csv"

# ── 检查所有文件是否存在 ────────────────────────────────────────────
all_files = {
    "Taxi Parquet"  : TAXI_PATH,
    "Taxi Zones"    : TAXI_ZONES_DIR,
    "Events CSV"    : EVENTS_PATH,
    "Reddit CSV"    : REDDIT_PATH,
    "Twitter CSV"   : TWITTER_PATH,
}

print("📁 Dataset File Check:")
all_ok = True
for name, path in all_files.items():
    exists = path.exists()
    status = "✅" if exists else "❌ NOT FOUND"
    print(f"  {status}  {name}: {path}")
    if not exists:
        all_ok = False

print()
print("✅ All files found!" if all_ok else "⚠️  Some files missing — update paths above.")

## 3. 出租车数据加载与探索 (NYC Yellow Taxi Data)

In [ ]:
# ── 加载 Parquet ────────────────────────────────────────────────────
print("Loading taxi data...")
taxi = pd.read_parquet(TAXI_PATH)

print(f"\n📊 Taxi Dataset Overview")
print(f"  Rows    : {len(taxi):,}")
print(f"  Columns : {taxi.shape[1]}")
print(f"  Memory  : {taxi.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nColumn names:")
print(taxi.columns.tolist())

In [ ]:
# ── 基础统计 ────────────────────────────────────────────────────────
print("\n--- First 5 rows ---")
display(taxi.head())

print("\n--- Data Types ---")
print(taxi.dtypes)

print("\n--- Missing Values ---")
missing = taxi.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values ✅")

In [ ]:
# ── 时间特征工程 ────────────────────────────────────────────────────
taxi["tpep_pickup_datetime"] = pd.to_datetime(taxi["tpep_pickup_datetime"])
taxi["tpep_dropoff_datetime"] = pd.to_datetime(taxi["tpep_dropoff_datetime"])

taxi["pickup_hour"]    = taxi["tpep_pickup_datetime"].dt.hour
taxi["pickup_dow"]     = taxi["tpep_pickup_datetime"].dt.dayofweek   # 0=Mon
taxi["pickup_date"]    = taxi["tpep_pickup_datetime"].dt.date
taxi["trip_duration"]  = (
    taxi["tpep_dropoff_datetime"] - taxi["tpep_pickup_datetime"]
).dt.total_seconds() / 60  # minutes

# 按小时聚合行程量（这是模型的核心目标变量）
hourly_demand = (
    taxi.groupby(taxi["tpep_pickup_datetime"].dt.floor("h"))
    .size()
    .reset_index(name="trip_count")
)
hourly_demand.columns = ["datetime", "trip_count"]

print(f"Hourly demand shape: {hourly_demand.shape}")
display(hourly_demand.head(10))

In [ ]:
# ── 可视化 1: 每小时行程量分布 ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 时序图
axes[0].plot(hourly_demand["datetime"], hourly_demand["trip_count"],
             linewidth=0.8, color="steelblue", alpha=0.8)
axes[0].set_title("Hourly Taxi Demand – January 2026", fontsize=13)
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Trip Count")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
axes[0].tick_params(axis='x', rotation=45)

# 按小时平均需求（0–23时）
avg_by_hour = taxi.groupby("pickup_hour").size() / taxi["pickup_date"].nunique()
axes[1].bar(avg_by_hour.index, avg_by_hour.values, color="steelblue", edgecolor="white")
axes[1].set_title("Average Demand by Hour of Day", fontsize=13)
axes[1].set_xlabel("Hour of Day")
axes[1].set_ylabel("Avg Trip Count")
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig("../results/taxi_demand_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved to results/taxi_demand_overview.png")

In [ ]:
# ── 可视化 2: 按星期几的需求热力图 ─────────────────────────────────
dow_hour = taxi.groupby(["pickup_dow", "pickup_hour"]).size().unstack(fill_value=0)
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

plt.figure(figsize=(14, 4))
sns.heatmap(dow_hour, cmap="YlOrRd", linewidths=0.3,
            yticklabels=dow_labels, cbar_kws={"label": "Trip Count"})
plt.title("Taxi Demand Heatmap: Day of Week × Hour of Day", fontsize=13)
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.savefig("../results/demand_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved to results/demand_heatmap.png")

## 4. 地理数据加载 (Taxi Zone Shapefile)

In [ ]:
# ── 加载 Shapefile ──────────────────────────────────────────────────
shp_file = list(TAXI_ZONES_DIR.glob("*.shp"))[0]  # 自动找 .shp 文件
zones = gpd.read_file(shp_file)

print(f"Taxi Zones loaded: {len(zones)} zones")
print(f"Columns: {zones.columns.tolist()}")
display(zones.head(3))

# 按区域统计行程量
pickup_by_zone = taxi["PULocationID"].value_counts().reset_index()
pickup_by_zone.columns = ["LocationID", "trip_count"]

zones_merged = zones.merge(pickup_by_zone, on="LocationID", how="left")
zones_merged["trip_count"] = zones_merged["trip_count"].fillna(0)

# 地图可视化
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
zones_merged.plot(
    column="trip_count", cmap="hot_r", linewidth=0.3,
    edgecolor="grey", legend=True,
    legend_kwds={"label": "Pickup Count", "shrink": 0.6},
    ax=ax
)
ax.set_title("NYC Taxi Pickup Density by Zone – January 2026", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.savefig("../results/taxi_zone_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Map saved to results/taxi_zone_map.png")

## 5. 社交媒体情感数据 (Reddit + Twitter)

In [ ]:
# ── 加载 Reddit 数据 ────────────────────────────────────────────────
reddit = pd.read_csv(REDDIT_PATH)
print("=== Reddit Data ===")
print(f"Shape: {reddit.shape}")
print(f"Columns: {reddit.columns.tolist()}")
display(reddit.head(3))

print(f"\nMissing values:")
print(reddit.isnull().sum())

In [ ]:
# ── 加载 Twitter 数据 ───────────────────────────────────────────────
twitter = pd.read_csv(TWITTER_PATH)
print("=== Twitter Data ===")
print(f"Shape: {twitter.shape}")
print(f"Columns: {twitter.columns.tolist()}")
display(twitter.head(3))

print(f"\nMissing values:")
print(twitter.isnull().sum())

In [ ]:
# ── 情感标签分布可视化 ──────────────────────────────────────────────
# 注意：根据你的实际列名修改 'category' 或 'label' 字段
# 以下假设情感列名为 'category'（Kaggle 该数据集的标准列名）

sentiment_col = "category"   # ← 如果列名不同，在这里修改

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (df, name) in zip(axes, [(reddit, "Reddit"), (twitter, "Twitter")]):
    if sentiment_col in df.columns:
        counts = df[sentiment_col].value_counts()
        colors = ["#2ecc71", "#e74c3c", "#95a5a6"][:len(counts)]
        ax.bar(counts.index.astype(str), counts.values, color=colors, edgecolor="white")
        ax.set_title(f"{name} Sentiment Distribution", fontsize=12)
        ax.set_xlabel("Sentiment")
        ax.set_ylabel("Count")
        for i, v in enumerate(counts.values):
            ax.text(i, v + 50, f"{v:,}", ha="center", fontsize=9)
    else:
        ax.text(0.5, 0.5, f"Column '{sentiment_col}' not found\nPlease update column name",
                ha="center", va="center", transform=ax.transAxes, fontsize=11)
        ax.set_title(f"{name} – column check needed")

plt.tight_layout()
plt.savefig("../results/sentiment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved to results/sentiment_distribution.png")

## 6. 城市事件数据 (NYC Special Events)

In [ ]:
# ── 加载 Events CSV ─────────────────────────────────────────────────
events = pd.read_csv(EVENTS_PATH)

print(f"NYC Events Dataset: {events.shape}")
print(f"Columns: {events.columns.tolist()}")
display(events.head(5))

print(f"\nEvent types:")
# 找事件类型列（常见列名）
type_cols = [c for c in events.columns if any(
    kw in c.lower() for kw in ["type", "category", "event"])]
if type_cols:
    print(events[type_cols[0]].value_counts().head(10))

In [ ]:
# ── 节假日标签生成 ──────────────────────────────────────────────────
ny_holidays = holidays.US(state="NY", years=2026)

# 为 hourly_demand 添加节假日标签
hourly_demand["date"] = pd.to_datetime(hourly_demand["datetime"]).dt.date
hourly_demand["is_holiday"] = hourly_demand["date"].apply(
    lambda d: 1 if d in ny_holidays else 0
)
hourly_demand["day_of_week"] = pd.to_datetime(hourly_demand["datetime"]).dt.dayofweek
hourly_demand["hour"] = pd.to_datetime(hourly_demand["datetime"]).dt.hour
hourly_demand["is_weekend"] = (hourly_demand["day_of_week"] >= 5).astype(int)

print("January 2026 NY Holidays:")
for date, name in sorted(ny_holidays.items()):
    if date.year == 2026 and date.month == 1:
        print(f"  {date}: {name}")

print(f"\nHoliday hours flagged: {hourly_demand['is_holiday'].sum()}")
display(hourly_demand.head(8))

## 7. 天气数据获取 (Open-Meteo API)

In [ ]:
# ── 获取 NYC 1月逐小时天气数据 ─────────────────────────────────────
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 40.7128,
    "longitude": -74.0060,
    "start_date": "2026-01-01",
    "end_date": "2026-01-31",
    "hourly": "temperature_2m,precipitation,windspeed_10m,weathercode",
    "timezone": "America/New_York"
}

try:
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    wdata = resp.json()["hourly"]

    weather = pd.DataFrame({
        "datetime"       : pd.to_datetime(wdata["time"]),
        "temperature_c"  : wdata["temperature_2m"],
        "precipitation_mm": wdata["precipitation"],
        "windspeed_kmh"  : wdata["windspeed_10m"],
        "weather_code"   : wdata["weathercode"]
    })

    weather.to_csv("../data/weather_nyc_2026_01.csv", index=False)
    print(f"✅ Weather data loaded: {weather.shape}")
    display(weather.head(5))

except Exception as e:
    print(f"⚠️  API call failed: {e}")
    print("Creating mock weather data for offline development...")
    hours = pd.date_range("2026-01-01", "2026-01-31 23:00", freq="h")
    weather = pd.DataFrame({
        "datetime"        : hours,
        "temperature_c"   : np.random.normal(2, 5, len(hours)),
        "precipitation_mm": np.random.exponential(0.3, len(hours)),
        "windspeed_kmh"   : np.random.normal(15, 5, len(hours)).clip(0),
        "weather_code"    : np.random.choice([0, 1, 2, 3, 61, 71], len(hours))
    })
    print(f"Mock weather data created: {weather.shape}")

## 8. 多模态特征合并 (Feature Fusion)

In [ ]:
# ── 合并 taxi + weather + holiday ──────────────────────────────────
hourly_demand["datetime"] = pd.to_datetime(hourly_demand["datetime"])
weather["datetime"] = pd.to_datetime(weather["datetime"])

dataset = hourly_demand.merge(weather, on="datetime", how="left")

print(f"Merged dataset shape: {dataset.shape}")
print(f"Columns: {dataset.columns.tolist()}")
display(dataset.head(5))

print(f"\nMissing values after merge:")
print(dataset.isnull().sum())

In [ ]:
# ── 可视化: 特征相关性热力图 ────────────────────────────────────────
numeric_cols = ["trip_count", "temperature_c", "precipitation_mm",
                "windspeed_kmh", "hour", "day_of_week", "is_weekend", "is_holiday"]
corr = dataset[numeric_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            mask=mask, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.savefig("../results/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved to results/correlation_matrix.png")

In [ ]:
# ── 保存合并后的特征数据集 ──────────────────────────────────────────
dataset.to_csv("../data/merged_features_2026_01.csv", index=False)
print("✅ Merged dataset saved to data/merged_features_2026_01.csv")

## 9. 数据摘要报告 (Summary Report)

In [ ]:
print("=" * 60)
print("  MULTIMODAL URBAN EVENT PREDICTION – DATA SUMMARY")
print("=" * 60)
print(f"""
📦 Datasets Loaded:
  1. NYC Yellow Taxi (Jan 2026)  : {len(taxi):>10,} rows
  2. Taxi Zone Shapefile         : {len(zones):>10,} zones
  3. NYC Special Events          : {len(events):>10,} events
  4. Reddit Sentiment Data       : {len(reddit):>10,} posts
  5. Twitter Sentiment Data      : {len(twitter):>10,} tweets
  6. Weather Data (hourly NYC)   : {len(weather):>10,} hours

🔧 Features in Merged Dataset:
  - Temporal : hour, day_of_week, is_weekend, is_holiday
  - Weather  : temperature_c, precipitation_mm, windspeed_kmh
  - Target   : trip_count (hourly demand)
  - Pending  : social media sentiment score (next sprint)

📊 Results saved to: results/
  ✅ taxi_demand_overview.png
  ✅ demand_heatmap.png
  ✅ taxi_zone_map.png
  ✅ sentiment_distribution.png
  ✅ correlation_matrix.png

🚀 Environment: READY for model training
""")
print("=" * 60)